In [0]:
%sql
-- 1. Check for duplicate transaction IDs
SELECT COUNT(*) AS total_rows,
       COUNT(DISTINCT transaction_id) AS distinct_ids,
       COUNT(*) - COUNT(DISTINCT transaction_id) AS duplicate_count
FROM retail.sales.transactions_raw;

In [0]:

%sql

--2. Counter-example — MERGE dirty source

--This is intentionally expected to fail if transactions_raw contains duplicate transaction_id values.



MERGE INTO retail.sales.transactions AS target
USING retail.sales.transactions_raw AS source
ON target.transaction_id = source.transaction_id

WHEN MATCHED THEN
    UPDATE SET
        target.total_amount = source.total_amount

WHEN NOT MATCHED THEN
    INSERT (
        transaction_id,
        customer_id,
        product_id,
        store_id,
        transaction_date,
        quantity,
        unit_price,
        total_amount
    )
    VALUES (
        source.transaction_id,
        source.customer_id,
        source.product_id,
        source.store_id,
        source.transaction_date,
        source.quantity,
        source.unit_price,
        source.total_amount
    );

In [0]:

#3. Deduplicate the source

# Here the duplicate rows are identical, so dropDuplicates() is sufficient.

from pyspark.sql.functions import col

df_raw = spark.table("retail.sales.transactions_raw")

print(f"Rows before dedup: {df_raw.count()}")

df_deduped = df_raw.dropDuplicates(["transaction_id"])

print(f"Rows after dedup: {df_deduped.count()}")

display(df_deduped)

In [0]:
#4. Create a temporary view
df_deduped.createOrReplaceTempView("transactions_raw_clean")

#Now the cleaned DataFrame can be accessed from SQL as: transactions_raw_clean

In [0]:
%sql

select * from transactions_raw_clean;

In [0]:
%sql

--5. MERGE the clean source into the target

--This example inserts only transactions that do not already exist.



MERGE INTO retail.sales.transactions AS target
USING transactions_raw_clean AS source
ON target.transaction_id = source.transaction_id

WHEN NOT MATCHED THEN
    INSERT (
        transaction_id,
        customer_id,
        product_id,
        store_id,
        transaction_date,
        quantity,
        unit_price,
        total_amount
    )
    VALUES (
        source.transaction_id,
        source.customer_id,
        source.product_id,
        source.store_id,
        source.transaction_date,
        source.quantity,
        source.unit_price,
        source.total_amount
    );

In [0]:
%sql

--6. Verify the final row count


SELECT COUNT(*) AS transaction_count_after
FROM retail.sales.transactions;